# 03 — Neural-network implementations: PyTorch and TensorFlow/Keras

This notebook runs two implementations of the same small multilayer perceptron on the scikit-learn Diabetes regression benchmark. Both receive exactly the same training, validation and held-out test rows. The test partition is not used for fitting, early stopping or model selection.

The dataset is a small, preprocessed benchmark. This is an educational regression experiment, not a clinical model.

## Install the optional neural dependencies

In a terminal, install the base project and optional neural frameworks:

~~~bash
pip install -r requirements.txt
pip install -r requirements-neural.txt
pip install -e .
~~~

In [ ]:
import pandas as pd
from diabetes_ml.data import load_diabetes_frame, make_train_test_split
from diabetes_ml.neural.neural_data import prepare_neural_data

X, y = load_diabetes_frame()
shared_split = make_train_test_split(X, y, random_state=42)
neural_data = prepare_neural_data(shared_split, random_state=42)

print('Input feature count:', X.shape[1])
print('Neural training rows:', len(neural_data.X_train))
print('Neural validation rows:', len(neural_data.X_validation))
print('Held-out test rows:', len(neural_data.X_test))

## Architecture

Both models use 10 input features, dense layers with 16 and 8 ReLU units, and one linear output. They use Adam, mean-squared-error loss and validation-loss early stopping. This architecture is intentionally small because the benchmark has only 442 rows.

In [ ]:
from diabetes_ml.neural.pytorch_model import train_pytorch_model

pytorch_result = train_pytorch_model(
    neural_data, epochs=200, patience=20, batch_size=32, seed=42
)
print('PyTorch metrics:', pytorch_result.metrics)
print('PyTorch best epoch:', pytorch_result.best_epoch)

In [ ]:
from diabetes_ml.neural.tensorflow_model import train_tensorflow_model

tensorflow_result = train_tensorflow_model(
    neural_data, epochs=200, patience=20, batch_size=32, seed=42
)
print('TensorFlow metrics:', tensorflow_result.metrics)
print('TensorFlow best epoch:', tensorflow_result.best_epoch)

In [ ]:
comparison = pd.DataFrame(
    [
        {'framework': 'PyTorch', **pytorch_result.metrics},
        {'framework': 'TensorFlow/Keras', **tensorflow_result.metrics},
    ]
)
display(comparison)

## Interpretation checklist

- Compare training and validation curves; check for overfitting.
- Compare both neural implementations with conventional models in model_metrics.csv.
- Interpret MAE and RMSE in the target's units; R-squared can be negative on a held-out test set.
- Do not assume a neural network should outperform a regularised linear model on this small dataset.
- Do not interpret this benchmark as clinical validation or a model for individual patient care.